# ACA N° 2 – Computación en la Nube
## Chatbot RAG de orientación estudiantil con Ollama (phi3), Chroma y Streamlit

**CUN · Especialización en Inteligencia Artificial** · Docente: Dr. Hamilton Rivera Flor
**Pilar:** Proyecto 2 – Educación · **Integrantes:** [Nombre 1], [Nombre 2], [Nombre 3]

**Pregunta orientadora.** ¿Cómo articular un modelo de lenguaje con una base de datos vectorial para construir un
chatbot RAG confiable, con una interfaz gráfica que facilite la administración documental, la consulta en lenguaje
natural y la reducción de alucinaciones con respuestas fundamentadas en evidencia?

### Arquitectura

| Capa | Tecnología | Archivo |
|---|---|---|
| Corpus | Reglamento estudiantil (PDF) + documentos generados desde xAPI-Edu-Data + resultados del modelo MLP | `corpus.py`, `docs/`, `data/` |
| Base vectorial | Chroma persistente en disco, distancia coseno | `rag_core.py`, `chroma_db/` |
| LLM | Ollama local con **phi3** | `rag_core.py`, `.env` |
| Embeddings | Ollama local con **nomic-embed-text** | `rag_core.py`, `.env` |
| Orquestación RAG | LangChain: recuperación top-k → prompt con contexto numerado → respuesta con citas | `rag_core.py` |
| Interfaz | Streamlit: chat con historial + gestión documental | `app.py` |
| Evaluación | Casos de prueba, fidelidad, latencia, robustez | `evaluacion.py`, `pruebas/` |

**Flujo de una consulta:** pregunta → embedding (nomic-embed-text) → búsqueda top-k en Chroma → plantilla de prompt con los
fragmentos numerados → phi3 genera la respuesta citando [n] → la app muestra respuesta, fuentes y tiempos.

> Este proyecto continúa el ACA 1 (`CHAT_ACA1.py`): se conserva la pila LangChain + Chroma + Ollama + Streamlit y se
> añaden persistencia, gestión de credenciales, citas, gestión documental y evaluación.

### Requisitos previos
1. Instalar Ollama (https://ollama.com) y descargar los modelos: `ollama pull phi3` y `ollama pull nomic-embed-text`
2. `pip install -r requirements.txt`
3. Copiar `.env.example` como `.env`

In [ ]:
import json, time, warnings
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 140)

import corpus, rag_core, construir_indice, evaluacion
cfg = rag_core.Config()
print("Configuración cargada desde .env")

---
# PASO 1 · Conexión y recuperación de la base vectorial

En el ACA 1 la base vectorial se creaba en memoria a partir de un PDF cada vez que arrancaba la app. Ahora la base
**Chroma queda persistente en disco** (`chroma_db/`): se indexa una vez y la app y este notebook se conectan a ella.

### 1.1 Corpus documental

In [ ]:
docs_csv = corpus.documentos_desde_csv(incluir_fichas=cfg.incluir_fichas)
docs_modelo = corpus.documento_modelo()
docs_pdf = corpus.documentos_externos()
todos = docs_csv + docs_modelo + docs_pdf
tabla = pd.DataFrame([{"fuente": d.metadata["fuente"], "tipo": d.metadata["tipo"],
                       "articulo": d.metadata.get("articulo", 0), "caracteres": len(d.page_content)} for d in todos])
print(f"{len(todos)} secciones · {tabla.fuente.nunique()} documentos")
tabla.groupby(["tipo", "fuente"]).agg(secciones=("fuente", "size"), caracteres=("caracteres", "sum"))

In [ ]:
# Ejemplo: un artículo del reglamento y un documento generado desde el dataset
print(docs_pdf[18].metadata, "\n", docs_pdf[18].page_content, "\n")
print(next(d for d in docs_csv if d.metadata["doc_id"] == "factores_riesgo").page_content)

### 1.2 Verificación del servidor Ollama y construcción del índice

In [ ]:
estado = rag_core.verificar_ollama(cfg)
print(json.dumps(estado, indent=2, ensure_ascii=False))
assert estado["ok"], f"Ollama no está listo. Ejecuta: ollama serve, ollama pull {cfg.llm_model} y ollama pull {cfg.embedding_model}"

In [ ]:
vs = rag_core.conectar_vectorstore(cfg)
if vs._collection.count() == 0:
    print("Base vacía: indexando el corpus (con phi3 puede tardar algunos minutos)…")
    print(construir_indice.construir(cfg, vs=vs))
print("Fragmentos en la colección:", vs._collection.count())
rag_core.resumen_documentos(vs)

### 1.3 Recuperación semántica de los Top-k fragmentos

In [ ]:
for consulta in ["requisitos para homologar una asignatura",
                 "límite de ausencias en el semestre",
                 "relación entre ausencias y bajo rendimiento"]:
    print(f"\nConsulta: {consulta}")
    for i, (d, s) in enumerate(rag_core.recuperar(vs, consulta, cfg.top_k), 1):
        print(f"  [{i}] relevancia {s:.3f} · {rag_core.etiqueta_fuente(d)}")

---
# PASO 2 · Selección y configuración del LLM (Ollama local)

**Modelo elegido: phi3 (Microsoft, 3,8 mil millones de parámetros) servido con Ollama.**

| Ventaja de la opción local | Detalle |
|---|---|
| Costo cero | No hay cobro por token ni cuotas de API |
| Privacidad | Los datos de estudiantes no salen del equipo |
| Sin credenciales externas | No hay API keys que puedan filtrarse |
| Continuidad | Es el mismo modelo del ACA 1 |
| Limitación | Más lento que una API en la nube y con menos capacidad de razonamiento que modelos grandes |

**Modelo de embeddings: nomic-embed-text (también en Ollama).** En el ACA 1 los embeddings se generaban con phi3.
Las versiones recientes de Ollama solo generan embeddings con modelos especializados (con phi3 responden
*"This server does not support embeddings"*). Además, un modelo entrenado para embeddings produce vectores que
representan mejor el significado del texto, lo que mejora la recuperación. nomic-embed-text pesa unos 270 MB,
genera vectores de 768 dimensiones y corre en CPU.

**Hiperparámetros (en `.env`)**

| Parámetro | Valor | Justificación |
|---|---|---|
| `temperature` | 0.2 | En el ACA 1 era 0.7. Un valor bajo hace las respuestas más deterministas y fieles al contexto, lo que reduce alucinaciones |
| `top_p` | 0.9 | Descarta tokens muy improbables sin volver rígida la redacción |
| `num_ctx` | 4096 | Alcanza para el prompt del sistema, 4 fragmentos de ~700 caracteres y 3 turnos de historial |
| `num_predict` | 512 | Limita la longitud de la respuesta y el tiempo de generación |
| `top_k` (recuperación) | 4 | Equilibrio entre cubrir la respuesta y no saturar el contexto (se evalúa en el paso 5) |

**Gestión segura de configuración:** todo se lee con `python-dotenv` desde `.env`, que está en `.gitignore`.
En el repositorio solo se publica `.env.example`. Si se cambiara a una API en la nube, la clave iría como
`GOOGLE_API_KEY` en `.env` o en los *secrets* de Streamlit, nunca en el código.

In [ ]:
# Configuración efectiva (sin mostrar secretos: este proyecto no usa claves de API)
{k: v for k, v in vars(cfg).items() if "key" not in k.lower()}

In [ ]:
llm = rag_core.crear_llm(cfg)
t0 = time.perf_counter()
prueba = llm.invoke("Responde en una sola frase en español: ¿qué es un chatbot RAG?")
print(prueba.content.strip())
print(f"\nLatencia de una llamada directa: {time.perf_counter() - t0:.1f} s")

In [ ]:
# Verificación de seguridad: .env no se versiona y no hay claves escritas en el código
gitignore = (rag_core.BASE / ".gitignore").read_text()
print(".env ignorado por git:", ".env" in gitignore.splitlines())
import re, pathlib
sospechosos = [p.name for p in pathlib.Path(rag_core.BASE).glob("*.py")
               if re.search(r"(api_key|API_KEY)\s*=\s*['\"][A-Za-z0-9_\-]{12,}", p.read_text(encoding="utf-8"))]
print("Archivos con posibles claves en el código:", sospechosos or "ninguno")

---
# PASO 3 · Canalización RAG y prompt engineering

La plantilla combina tres piezas:
1. **Prompt del sistema**: rol (Sofía, orientadora), alcance y reglas anti-alucinación: responder solo con el
   contexto, citar `[n]` y el artículo, decir *"No encuentro esa información…"* cuando falte, y redirigir preguntas
   fuera de tema. En el ACA 1 el prompt pedía ocultar que era una IA; aquí se eliminó esa instrucción por transparencia.
2. **Contexto numerado**: cada fragmento recuperado va con su número y su fuente (documento, artículo, página).
3. **Memoria conversacional**: los últimos 3 turnos, sin repetir los contextos anteriores para no saturar a phi3.

In [ ]:
print(rag_core.SYSTEM_PROMPT)
print("\n--- Plantilla del mensaje del usuario ---\n" + rag_core.PLANTILLA_USUARIO)

In [ ]:
pregunta = "¿Qué requisitos debo cumplir para homologar una asignatura?"
frag = rag_core.recuperar(vs, pregunta, cfg.top_k)
mensajes = rag_core.construir_mensajes(pregunta, frag, [], cfg)
print(mensajes[-1].content[:1500])

In [ ]:
def conversar(pregunta, historial=None):
    r = rag_core.responder(pregunta, vs, llm, cfg, historial=historial or [])
    print(f"PREGUNTA: {pregunta}\n\nRESPUESTA:\n{r['respuesta']}\n\nFUENTES:")
    usadas = rag_core.citas_usadas(r["respuesta"])
    for i, (d, s) in enumerate(r["fuentes"], 1):
        print(f"  [{i}] {'(citada) ' if i in usadas else ''}{rag_core.etiqueta_fuente(d)} · relevancia {s:.2f}")
    print(f"\nTiempo: recuperación {r['t_recuperacion']*1000:.0f} ms · generación {r['t_generacion']:.1f} s")
    return r

r1 = conversar("¿Qué requisitos debo cumplir para homologar una asignatura?")

In [ ]:
r2 = conversar("¿Qué porcentaje de estudiantes con más de 7 ausencias tiene bajo rendimiento y qué dice el reglamento sobre esas ausencias?")

In [ ]:
# Diálogo de varios turnos: la segunda pregunta depende de la primera
h = [{"role": "user", "content": "¿Cómo se calcula la nota final?"}]
r3 = conversar("¿Cómo se calcula la nota final?")
h.append({"role": "assistant", "content": r3["respuesta"]})
r4 = conversar("¿Y si me va mal, puedo recuperarla?", historial=h)

In [ ]:
r5 = conversar("¿Quién ganó el último mundial de fútbol?")  # fuera de contexto: debe abstenerse

---
# PASO 4 · Interfaz gráfica (Streamlit)

`app.py` parte del código del ACA 1 e incorpora:

**a) Módulo de chat**
- Historial conversacional con avatares propios (búho = Sofía, estudiante = usuario) y portada temática.
- Preguntas sugeridas para empezar.
- Cada respuesta muestra las **fuentes citadas** (documento, artículo, página, relevancia) y los tiempos de
  recuperación y generación.
- Ajustes en la barra lateral: top-k y filtro por tipo de documento; estado de Ollama y parámetros del modelo.

**b) Módulo de gestión documental**
- Métricas de la colección (documentos, fragmentos, modelo de embeddings).
- Tabla de documentos indexados con tipo, fragmentos y páginas.
- Explorador de fragmentos y metadatos por documento.
- Prueba de recuperación semántica con barra de relevancia.
- Carga de nuevos PDF/TXT, eliminación de documentos y reconstrucción del índice.

**Ejecución:** `streamlit run app.py` → http://localhost:8501

In [ ]:
from IPython.display import Image, display
display(Image(filename="assets/banner.png", width=800))

---
# PASO 5 · Validación integral y optimización

El banco de pruebas (`pruebas/casos_prueba.csv`) tiene 19 casos en cuatro categorías: preguntas del reglamento,
preguntas sobre los datos, preguntas complejas que combinan ambas fuentes y preguntas fuera de contexto.

| Métrica | Qué mide |
|---|---|
| Acierto de recuperación | El documento y el artículo esperados están entre los top-k fragmentos (precisión contextual) |
| Cobertura | Proporción de datos clave esperados presentes en la respuesta (fidelidad) |
| Cita | La respuesta incluye al menos una referencia [n] |
| Abstención | Reconoce que no tiene la información (lo esperado fuera de contexto) |
| Latencia | Tiempo de recuperación, de generación y total |

In [ ]:
casos = evaluacion.cargar_casos()
casos[["id", "categoria", "pregunta", "fuente_esperada", "articulo_esperado", "palabras_clave"]]

### 5.1 Precisión de la recuperación según top-k (optimización)

In [ ]:
rec = pd.concat([evaluacion.evaluar_recuperacion(vs, casos[casos.categoria != "fuera_de_contexto"], k) for k in [1, 2, 4, 6, 8]])
hit = rec.groupby("k").acierto_recuperacion.mean()
ax = hit.plot(marker="o", figsize=(6, 3.5), ylim=(0, 1.05), title="Acierto de recuperación según top-k")
ax.set_ylabel("Proporción de aciertos"); ax.grid(alpha=.3); plt.show()
hit.round(3)

### 5.2 Evaluación de extremo a extremo (recuperación + generación)

In [ ]:
resultados = evaluacion.evaluar_respuestas(vs, llm, cfg, casos)
resultados.to_csv("pruebas/resultados_evaluacion.csv", index=False)
evaluacion.resumen(resultados)

In [ ]:
resultados[["id", "correcto", "cobertura", "cita", "abstencion", "t_total_s", "respuesta"]]

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
evaluacion.resumen(resultados)["correctos"].plot(kind="bar", ax=ax[0], color="#0B7A70", ylim=(0, 1.05), rot=0)
ax[0].set_title("Respuestas correctas por categoría"); ax[0].set_ylabel("Proporción")
resultados.boxplot(column="t_total_s", by="categoria", ax=ax[1], grid=False)
ax[1].set_title("Tiempo total de respuesta (s)"); ax[1].set_xlabel(""); plt.suptitle("")
plt.tight_layout(); plt.show()
print(f"Latencia media: {resultados.t_total_s.mean():.1f} s · recuperación media {resultados.t_recuperacion_s.mean()*1000:.0f} ms")

### 5.3 Robustez ante preguntas fuera de contexto: calibración del umbral de relevancia
Si la relevancia máxima de las preguntas fuera de contexto es claramente menor que la de las preguntas válidas,
se puede fijar `UMBRAL_RELEVANCIA` en `.env`: por debajo de ese valor el sistema responde que no tiene la información
sin llamar al LLM, lo que ahorra tiempo y evita alucinaciones.

In [ ]:
dentro = resultados[resultados.categoria != "fuera_de_contexto"].relevancia_max
fuera = resultados[resultados.categoria == "fuera_de_contexto"].relevancia_max
plt.figure(figsize=(6, 3.5))
plt.hist(dentro, bins=12, alpha=.7, label="Preguntas válidas", color="#0B7A70")
plt.hist(fuera, bins=6, alpha=.7, label="Fuera de contexto", color="#E4572E")
plt.xlabel("Relevancia máxima recuperada"); plt.legend(); plt.title("Separación entre preguntas válidas y fuera de contexto")
plt.show()
umbral = round((dentro.min() + fuera.max()) / 2, 3) if dentro.min() > fuera.max() else None
print(f"Mínimo en válidas: {dentro.min():.3f} · máximo fuera de contexto: {fuera.max():.3f}")
print("Umbral sugerido:", umbral if umbral else "no hay separación clara; se deja la abstención en manos del prompt")

### 5.4 Conclusiones
Completa esta sección con los resultados obtenidos al ejecutar el notebook:
- **Precisión contextual:** porcentaje de aciertos de recuperación con top-k = 4 y el valor de k recomendado.
- **Fidelidad:** proporción de respuestas correctas con cita en las preguntas del reglamento y de datos.
- **Robustez:** cuántas preguntas fuera de contexto se rechazaron correctamente y si conviene activar el umbral.
- **Latencia:** tiempo medio de respuesta con phi3 en tu equipo y la parte que corresponde a la generación.
- **Mejoras posibles:** re-ranking de los fragmentos recuperados, búsqueda híbrida (palabras clave + vectores) o un
  modelo de lenguaje más grande si el hardware lo permite.

### Referencias
- Aljarah, I. (2016). *xAPI-Edu-Data (Students' Academic Performance)* [Conjunto de datos]. Kaggle. https://www.kaggle.com/datasets/aljarah/xAPI-Edu-Data
- Chase, H. (2022). *LangChain*. https://github.com/langchain-ai/langchain
- Lewis, P., et al. (2020). Retrieval-augmented generation for knowledge-intensive NLP tasks. *Advances in Neural Information Processing Systems, 33*, 9459–9474. https://arxiv.org/abs/2005.11401
- Abdin, M., et al. (2024). *Phi-3 technical report: A highly capable language model locally on your phone*. arXiv. https://arxiv.org/abs/2404.14219
- Streamlit Inc. (2024). *Streamlit documentation*. https://docs.streamlit.io/